# FloodShift 09B — Controlled Variational VLM Refinement

This notebook is the **single controlled refinement** after FloodShift-09A.

## Why 09A was not accepted

09A trained successfully, but failed three scientific gates:

- full vs shuffled visual conditioning was effectively identical;
- Gaussian latent uncertainty had near-zero correlation with FloodShift predictive entropy;
- free-form generation had weak numeric binding / unsupported-number errors.

09A is preserved as a baseline and **must not be overwritten**.

Recorded 09A archive SHA256:

```text
126bcac0ac4ae899ae6a3eb9cb4a47dad0ee6ac92cf22454d6737574713909cc
```

## 09B changes only

### A. Richer visual representation

09A used one pooled CLIP vector.

09B uses the frozen CLIP ViT spatial patch grid:

\[
7\times7\times768
\]

and deterministically pools it to:

\[
3\times3=9
\]

visual tokens.

### B. Two-stage Gaussian training

The Gaussian scene encoder is first trained **only at scene level** to encode:

- temporary-flood fraction,
- predictive entropy,
- optical attribution share,
- sensor dominance,
- fixed review decision.

Its posterior uncertainty is explicitly supervised with:

\[
L_U =
L_{\mathrm{MSE}}
+
\lambda_c(1-r)
+
\lambda_r L_{\mathrm{rank}}
\]

where \(r\) is batch Pearson correlation against the accepted calibrated predictive entropy.

After this stage, the Gaussian scene encoder is **frozen** so language training cannot destroy its uncertainty semantics.

### C. Task-specific deterministic evidence

Instead of giving FLAN-T5 every number for every question, each grounded task receives only the verified fields relevant to that task.

This reduces cross-field number swapping.

### D. Visual-only grounding tasks

Four deterministic visual tasks are added per scene:

- qualitative flood-extent bin,
- qualitative uncertainty bin,
- sensor-dependence dominance,
- fixed review flag.

These prompts contain **no structured scene facts**. They force the language decoder to use the visual pathway.

The bins are training/evaluation descriptors only; they are **not emergency risk thresholds**.

### E. Correct shuffled-vision ablation

09A rolled visual tokens inside a batch, which could leave the same scene paired with itself.

09B creates a **global scene-level derangement**:

\[
\text{scene}_i \rightarrow \text{scene}_{i+1}
\]

within the held-out validation scenes.

Thus shuffled vision always comes from a different scene.

## Test-set rule

The frozen Pakistan + Spain geographic test remains:

```text
LOCKED_NOT_EVALUATED
```

No test scene is read by this notebook.


In [ ]:
# ============================================================
# P19-C00 — KAGGLE DEPENDENCIES
# Run BEFORE importing PEFT.
# ============================================================
!pip uninstall -y torchao >/dev/null 2>&1
!pip -q install -U "transformers>=4.46,<5" "peft>=0.13,<1" accelerate sentencepiece safetensors

print("P19-C00 COMPLETE ✅")
print("torchao removed because FloodShift does not use TorchAO quantization.")


In [ ]:
# ============================================================
# P19-C01 — ENVIRONMENT
# FP32 is intentional because 09A showed FP16 instability.
# ============================================================
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, json, math, time, random, re, hashlib, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoImageProcessor,
    CLIPVisionModel,
    AutoModelForSeq2SeqLM,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 09A was unstable under T4 FP16. Keep the refinement numerically clean.
AMP_ENABLED = False

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_09B_Controlled_Refinement")
SOURCE_ROOT = OUTPUT_ROOT / "_source08b"
CACHE_ROOT = OUTPUT_ROOT / "cache"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoint"
METRIC_ROOT = OUTPUT_ROOT / "metrics"

for p in [
    OUTPUT_ROOT,
    SOURCE_ROOT,
    CACHE_ROOT,
    CHECKPOINT_ROOT,
    METRIC_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_08B_SHA = (
    "421bb74db41f90012b7e5d0f1b342ecfc467b657b12177806973f3bf2e13da20"
)

BASELINE_09A_SHA = (
    "126bcac0ac4ae899ae6a3eb9cb4a47dad0ee6ac92cf22454d6737574713909cc"
)

VISION_MODEL_NAME = "openai/clip-vit-base-patch32"
TEXT_MODEL_NAME = "google/flan-t5-base"

print("=" * 112)
print("P19-C01 — ENVIRONMENT")
print("=" * 112)
print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Device       :", DEVICE)
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
print("AMP          :", AMP_ENABLED)
print("09A baseline :", BASELINE_09A_SHA)
print("Geo test     : LOCKED_NOT_EVALUATED")
print("P19-C01 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C02 — LOCATE + VERIFY ACCEPTED 08B CORPUS
# ============================================================
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_all(filename):
    return sorted(
        INPUT_ROOT.rglob(filename),
        key=lambda p: (len(str(p)), str(p)),
    )

def find_one(filename, required=True):
    matches = find_all(filename)

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Could not find '{filename}' under /kaggle/input"
            )
        return None

    return matches[0]

archive08b = find_one(
    "FloodShift_08B_ARCHIVE.zip",
    required=False,
)

if archive08b is not None:
    found_sha = sha256_file(archive08b)

    assert found_sha == EXPECTED_08B_SHA, (
        "08B SHA mismatch\n"
        f"Expected: {EXPECTED_08B_SHA}\n"
        f"Found   : {found_sha}"
    )

    with zipfile.ZipFile(archive08b, "r") as zf:
        zf.extractall(SOURCE_ROOT)

    CORPUS_BASE = SOURCE_ROOT
    CORPUS_MODE = "verified_zip"

else:
    protocol_candidate = find_one(
        "FloodShift_08B_protocol_record.json"
    )

    CORPUS_BASE = protocol_candidate.parent
    CORPUS_MODE = "kaggle_auto_extracted"
    found_sha = EXPECTED_08B_SHA

def locate_corpus_file(filename):
    matches = list(CORPUS_BASE.rglob(filename))

    if matches:
        return sorted(
            matches,
            key=lambda p: (len(str(p)), str(p)),
        )[0]

    return find_one(filename)

TRAIN_JSONL = locate_corpus_file(
    "FloodShift_08B_train.jsonl"
)
VAL_JSONL = locate_corpus_file(
    "FloodShift_08B_validation.jsonl"
)
TRAIN_PACKETS_JSONL = locate_corpus_file(
    "FloodShift_08B_train_scene_packets.jsonl"
)
VAL_PACKETS_JSONL = locate_corpus_file(
    "FloodShift_08B_validation_scene_packets.jsonl"
)
DATASET_CARD_PATH = locate_corpus_file(
    "FloodShift_08B_dataset_card.json"
)
PROTOCOL_PATH = locate_corpus_file(
    "FloodShift_08B_protocol_record.json"
)
FILE_MANIFEST_PATH = locate_corpus_file(
    "FloodShift_08B_file_manifest.csv"
)

with open(DATASET_CARD_PATH, "r") as f:
    dataset_card = json.load(f)

with open(PROTOCOL_PATH, "r") as f:
    protocol08b = json.load(f)

assert dataset_card["training_scenes"] == 314
assert dataset_card["validation_scenes"] == 34
assert dataset_card["training_examples"] == 3140
assert dataset_card["validation_examples"] == 340
assert dataset_card["scene_overlap_train_validation"] == 0
assert dataset_card["test_status"] == "LOCKED_NOT_EVALUATED"
assert dataset_card["ground_truth_in_language_packets"] is False
assert protocol08b["05b_status"] == "NOT_PROMOTED_NOT_USED"
assert protocol08b["test_status"] == "LOCKED_NOT_EVALUATED"

manifest08b = pd.read_csv(FILE_MANIFEST_PATH)

manifest_failures = []

for _, row in manifest08b.iterrows():
    rel = Path(str(row["relative_path"]))
    expected = str(row["sha256"])

    candidate = CORPUS_BASE / rel

    if not candidate.exists():
        same_name = list(INPUT_ROOT.rglob(rel.name))

        if not same_name:
            manifest_failures.append((str(rel), "missing"))
            continue

        suffix_matches = [
            p for p in same_name
            if str(p).endswith(str(rel))
        ]

        candidate = (
            suffix_matches[0]
            if suffix_matches
            else same_name[0]
        )

    actual = sha256_file(candidate)

    if actual != expected:
        manifest_failures.append((str(rel), "sha_mismatch"))

assert not manifest_failures, (
    f"08B manifest failures: {manifest_failures[:10]}"
)

print("=" * 112)
print("P19-C02 — 08B VERIFIED")
print("=" * 112)
print("Mode              :", CORPUS_MODE)
print("Archive SHA       :", found_sha)
print("Manifest rows     :", len(manifest08b))
print("Manifest failures :", len(manifest_failures))
print("Geo test          : LOCKED_NOT_EVALUATED")
print("P19-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C03 — LOAD EXAMPLES / PACKETS / PANELS
# ============================================================
def read_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows

train_examples = read_jsonl(TRAIN_JSONL)
val_examples = read_jsonl(VAL_JSONL)
train_packets = read_jsonl(TRAIN_PACKETS_JSONL)
val_packets = read_jsonl(VAL_PACKETS_JSONL)

assert len(train_examples) == 3140
assert len(val_examples) == 340
assert len(train_packets) == 314
assert len(val_packets) == 34

packet_by_scene = {
    p["scene_id"]: p
    for p in train_packets + val_packets
}

train_scene_ids = sorted({
    x["scene_id"]
    for x in train_examples
})

val_scene_ids = sorted({
    x["scene_id"]
    for x in val_examples
})

assert len(train_scene_ids) == 314
assert len(val_scene_ids) == 34
assert set(train_scene_ids).isdisjoint(val_scene_ids)

all_images = list(
    CORPUS_BASE.rglob("*_evidence.png")
)

if len(all_images) < 348:
    all_images = list(
        INPUT_ROOT.rglob("*_evidence.png")
    )

image_by_scene = {}

for p in all_images:
    suffix = "_evidence.png"

    if not p.name.endswith(suffix):
        continue

    sid = p.name[:-len(suffix)]

    if sid in packet_by_scene and sid not in image_by_scene:
        image_by_scene[sid] = p

assert len(image_by_scene) == 348, (
    f"Expected 348 evidence panels, found {len(image_by_scene)}"
)

# Fixed scene-level derangements.
def make_derangement(scene_ids):
    ids = list(sorted(scene_ids))

    assert len(ids) > 1

    return {
        ids[i]: ids[(i + 1) % len(ids)]
        for i in range(len(ids))
    }

TRAIN_SHUFFLE_MAP = make_derangement(train_scene_ids)
VAL_SHUFFLE_MAP = make_derangement(val_scene_ids)

for sid in val_scene_ids:
    assert VAL_SHUFFLE_MAP[sid] != sid

print("=" * 112)
print("P19-C03 — CORPUS")
print("=" * 112)
print("Train scenes      :", len(train_scene_ids))
print("Validation scenes :", len(val_scene_ids))
print("Evidence panels   :", len(image_by_scene))
print("Scene overlap     :", len(set(train_scene_ids) & set(val_scene_ids)))
print("True derangement  : PASS")
print("P19-C03 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C04 — SCENE TARGETS + TASK-SPECIFIC VERIFIED EVIDENCE
# ============================================================
def review_label_from_examples(scene_id, split_examples):
    rows = [
        x for x in split_examples
        if x["scene_id"] == scene_id
        and x["task"] == "review_policy"
    ]

    assert len(rows) == 1

    answer = rows[0]["response"].strip().lower()

    if answer.startswith("yes"):
        return 1.0
    if answer.startswith("no"):
        return 0.0

    raise ValueError(
        f"Cannot parse review label for {scene_id}: {answer[:120]}"
    )

train_review = {
    sid: review_label_from_examples(sid, train_examples)
    for sid in train_scene_ids
}

val_review = {
    sid: review_label_from_examples(sid, val_examples)
    for sid in val_scene_ids
}

def normalize_dominance(value):
    t = str(value).lower()

    if "optical" in t:
        return "optical"
    if "sar" in t:
        return "sar"
    return "balanced"

DOM_TO_ID = {
    "balanced": 0,
    "optical": 1,
    "sar": 2,
}

ID_TO_DOM = {
    v: k
    for k, v in DOM_TO_ID.items()
}

def scene_targets(scene_id, split):
    p = packet_by_scene[scene_id]

    temp_fraction = float(
        p["segmentation"]["temporary_flood"]["fraction_of_coverage"]
    )
    entropy = float(
        p["uncertainty"]["mean_entropy"]
    )
    optical_share = float(
        p["counterfactual_attribution"]["optical_share"]
    )

    dominance = normalize_dominance(
        p["counterfactual_attribution"]["dominance"]
    )

    review = (
        train_review[scene_id]
        if split == "train"
        else val_review[scene_id]
    )

    return {
        "regression": np.asarray(
            [
                temp_fraction,
                entropy,
                optical_share,
            ],
            dtype=np.float32,
        ),
        "dominance": DOM_TO_ID[dominance],
        "review": float(review),
    }

def fmt_pct(x):
    if x is None:
        return "unavailable"
    return f"{100.0 * float(x):.2f}%"

def fmt_dec(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.3f}"

def fmt_area(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.2f} km²"

def flatten_scalars(obj, prefix=""):
    out = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            out.extend(
                flatten_scalars(v, key)
            )
        return out

    if obj is None:
        return [(prefix, "unavailable")]

    if isinstance(obj, (str, bool, int, float, np.integer, np.floating)):
        return [(prefix, str(obj))]

    return []

def task_evidence(example):
    p = packet_by_scene[
        example["scene_id"]
    ]

    task = example["task"]

    seg = p["segmentation"]
    unc = p["uncertainty"]
    sq = p["sensor_quality"]
    att = p["counterfactual_attribution"]
    geo = p["spatial_geometry"]

    if task == "extent":
        fields = [
            ("temporary_flood_fraction",
             fmt_pct(seg["temporary_flood"]["fraction_of_coverage"])),
            ("temporary_flood_area",
             fmt_area(seg["temporary_flood"]["area_km2"])),
            ("permanent_water_fraction",
             fmt_pct(seg["permanent_water"]["fraction_of_coverage"])),
        ]

    elif task == "uncertainty":
        fields = [
            ("mean_predictive_entropy",
             fmt_dec(unc["mean_entropy"])),
            ("review10_fraction",
             fmt_pct(unc["review10_fraction"])),
            ("review20_fraction",
             fmt_pct(unc["review20_fraction"])),
            ("mean_pclear",
             fmt_dec(sq["mean_pclear"])),
            ("low_pclear_fraction",
             fmt_pct(sq["low_pclear_fraction"])),
        ]

    elif task == "sensor_attribution":
        fields = [
            ("sensor_dominance",
             str(att["dominance"])),
            ("optical_attribution_share",
             fmt_pct(att["optical_share"])),
            ("sar_attribution_share",
             fmt_pct(att["sar_share"])),
            ("flip_rate_no_optical",
             fmt_pct(att["flip_rate_no_optical"])),
            ("flip_rate_no_sar",
             fmt_pct(att["flip_rate_no_sar"])),
        ]

    elif task == "temporary_sensor_attribution":
        fields = [
            ("temporary_js_optical",
             fmt_dec(att["temporary_flood_js_optical"])),
            ("temporary_js_sar",
             fmt_dec(att["temporary_flood_js_sar"])),
        ]

    elif task == "spatial_geometry":
        # Keep only the deterministic spatial-geometry section.
        fields = flatten_scalars(
            geo
        )

    elif task == "integrated_summary":
        fields = [
            ("temporary_flood_fraction",
             fmt_pct(seg["temporary_flood"]["fraction_of_coverage"])),
            ("temporary_flood_area",
             fmt_area(seg["temporary_flood"]["area_km2"])),
            ("mean_predictive_entropy",
             fmt_dec(unc["mean_entropy"])),
            ("review10_fraction",
             fmt_pct(unc["review10_fraction"])),
            ("mean_pclear",
             fmt_dec(sq["mean_pclear"])),
            ("sensor_dominance",
             str(att["dominance"])),
            ("optical_attribution_share",
             fmt_pct(att["optical_share"])),
            ("sar_attribution_share",
             fmt_pct(att["sar_share"])),
        ]

    elif task == "review_policy":
        # Do NOT leak the yes/no target.
        fields = [
            ("review10_fraction",
             fmt_pct(unc["review10_fraction"])),
            ("review20_fraction",
             fmt_pct(unc["review20_fraction"])),
            ("high_fragility_tile_fraction",
             fmt_pct(att["high_fragility_tile_fraction"])),
            ("mean_predictive_entropy",
             fmt_dec(unc["mean_entropy"])),
        ]

    elif task in {
        "cause_refusal",
        "impact_refusal",
        "forecast_refusal",
    }:
        fields = [
            (
                "scope",
                "FloodShift contains model-derived flood extent, "
                "uncertainty, reliability, spatial geometry and "
                "counterfactual sensor-dependence evidence only."
            ),
            (
                "causality_rule",
                "Counterfactual sensor attribution is model dependence, "
                "not physical causality."
            ),
        ]

    else:
        raise KeyError(
            f"Unknown task: {task}"
        )

    return " ; ".join(
        f"{k}={v}"
        for k, v in fields
    )

SYSTEM_PREFIX = (
    "You are FloodShift's grounded scientific assistant. "
    "Use only the verified evidence supplied for the current task. "
    "Do not invent numerical values, physical causes, human impacts, "
    "or future forecasts. Counterfactual sensor attribution measures "
    "model dependence, not physical causality."
)

def build_grounded_source(example):
    return (
        SYSTEM_PREFIX
        + "\nTASK-EVIDENCE: "
        + task_evidence(example)
        + "\nQUESTION: "
        + example["instruction"]
    )

print("=" * 112)
print("P19-C04 — TASK-SPECIFIC EVIDENCE PREVIEW")
print("=" * 112)

for task in [
    "extent",
    "uncertainty",
    "sensor_attribution",
    "review_policy",
]:
    ex = next(
        x for x in train_examples
        if x["task"] == task
    )

    print("\n---", task, "---")
    print(build_grounded_source(ex)[:1300])

print("\nP19-C04 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C05 — FROZEN CLIP SPATIAL FEATURE CACHE
#
# CLIP-B/32 gives 7x7 patch tokens at 224x224.
# We pool them to 3x3 = 9 visual tokens.
# ============================================================
image_processor = AutoImageProcessor.from_pretrained(
    VISION_MODEL_NAME
)

vision_model = CLIPVisionModel.from_pretrained(
    VISION_MODEL_NAME
).to(DEVICE).eval()

for p in vision_model.parameters():
    p.requires_grad = False

@torch.inference_mode()
def encode_panel(path):
    with Image.open(path) as img:
        img = img.convert("RGB")

        batch = image_processor(
            images=img,
            return_tensors="pt",
        )

    pixel_values = batch["pixel_values"].to(DEVICE)

    out = vision_model(
        pixel_values=pixel_values,
        output_hidden_states=False,
        return_dict=True,
    )

    pooled = out.pooler_output[0].float()

    patch = out.last_hidden_state[
        0,
        1:,
        :
    ].float()

    n_patch, dim = patch.shape

    side = int(round(math.sqrt(n_patch)))

    assert side * side == n_patch, (
        f"Expected square CLIP patch grid, got {n_patch}"
    )

    grid = (
        patch
        .reshape(
            side,
            side,
            dim,
        )
        .permute(
            2,
            0,
            1,
        )
        .unsqueeze(0)
    )

    pooled_grid = F.adaptive_avg_pool2d(
        grid,
        output_size=(3, 3),
    )

    patch9 = (
        pooled_grid[0]
        .permute(
            1,
            2,
            0,
        )
        .reshape(
            9,
            dim,
        )
    )

    return (
        pooled.cpu(),
        patch9.cpu(),
    )

VISION_CACHE = {}

t0 = time.time()

for i, sid in enumerate(
    sorted(image_by_scene),
    start=1,
):
    pooled, patch9 = encode_panel(
        image_by_scene[sid]
    )

    assert torch.isfinite(pooled).all()
    assert torch.isfinite(patch9).all()

    VISION_CACHE[sid] = {
        "pooled": pooled,
        "patch9": patch9,
    }

    if (
        i == 1
        or i % 50 == 0
        or i == len(image_by_scene)
    ):
        print(
            f"[{i:03d}/{len(image_by_scene)}] CLIP spatial cache"
        )

VISION_DIM = int(
    next(iter(VISION_CACHE.values()))[
        "pooled"
    ].numel()
)

PATCH_COUNT = int(
    next(iter(VISION_CACHE.values()))[
        "patch9"
    ].shape[0]
)

CACHE_PATH = (
    CACHE_ROOT
    / "FloodShift_09B_CLIP_spatial_cache.pt"
)

torch.save(
    {
        "vision_model_name": VISION_MODEL_NAME,
        "vision_dim": VISION_DIM,
        "patch_count": PATCH_COUNT,
        "cache": VISION_CACHE,
    },
    CACHE_PATH,
)

del vision_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 112)
print("P19-C05 — CLIP SPATIAL CACHE")
print("=" * 112)
print("Scenes      :", len(VISION_CACHE))
print("Global dim  :", VISION_DIM)
print("Patch tokens:", PATCH_COUNT)
print("Minutes     :", round((time.time() - t0) / 60.0, 2))
print("P19-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C06 — GAUSSIAN SCENE ENCODER
# ============================================================
LATENT_DIM = 256
LOGVAR_MIN = -6.0
LOGVAR_MAX = 1.0

class GaussianSceneEncoder(nn.Module):
    def __init__(
        self,
        vision_dim,
        latent_dim=LATENT_DIM,
    ):
        super().__init__()

        # Use both pooled CLS and mean spatial-patch information.
        self.scene_proj = nn.Sequential(
            nn.LayerNorm(
                vision_dim * 2
            ),
            nn.Linear(
                vision_dim * 2,
                512,
            ),
            nn.GELU(),
            nn.Dropout(0.10),
        )

        self.mu = nn.Linear(
            512,
            latent_dim,
        )

        self.logvar = nn.Linear(
            512,
            latent_dim,
        )

        self.regression_head = nn.Sequential(
            nn.Linear(
                latent_dim,
                128,
            ),
            nn.GELU(),
            nn.Linear(
                128,
                3,
            ),
        )

        self.dominance_head = nn.Linear(
            latent_dim,
            3,
        )

        self.review_head = nn.Linear(
            latent_dim,
            1,
        )

    def forward(
        self,
        pooled,
        patch9,
        sample=True,
    ):
        patch_mean = patch9.mean(
            dim=1
        )

        scene_input = torch.cat(
            [
                pooled.float(),
                patch_mean.float(),
            ],
            dim=-1,
        )

        h = self.scene_proj(
            scene_input
        )

        mu = self.mu(
            h
        )

        logvar = self.logvar(
            h
        ).clamp(
            LOGVAR_MIN,
            LOGVAR_MAX,
        )

        if sample:
            std = torch.exp(
                0.5 * logvar
            )

            z = (
                mu
                + std
                * torch.randn_like(
                    std
                )
            )
        else:
            z = mu

        regression = torch.sigmoid(
            self.regression_head(
                mu
            )
        )

        dominance_logits = self.dominance_head(
            mu
        )

        review_logit = self.review_head(
            mu
        ).squeeze(-1)

        # Direct posterior-variance summary.
        # This remains tied to logvar itself.
        latent_uncertainty = torch.sigmoid(
            logvar.mean(
                dim=1
            )
        )

        return {
            "mu": mu,
            "logvar": logvar,
            "z": z,
            "regression": regression,
            "dominance_logits": dominance_logits,
            "review_logit": review_logit,
            "latent_uncertainty": latent_uncertainty,
        }

gaussian_encoder = GaussianSceneEncoder(
    VISION_DIM
).to(DEVICE)

print("=" * 112)
print("P19-C06 — GAUSSIAN SCENE ENCODER")
print("=" * 112)
print("Vision dim :", VISION_DIM)
print("Latent dim :", LATENT_DIM)
print(
    "Trainable :",
    f"{sum(p.numel() for p in gaussian_encoder.parameters() if p.requires_grad):,}"
)
print("P19-C06 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C07 — SCENE-LEVEL GAUSSIAN PRETRAIN DATA
# ============================================================
class SceneDataset(Dataset):
    def __init__(
        self,
        scene_ids,
        split,
    ):
        self.scene_ids = list(scene_ids)
        self.split = split

    def __len__(self):
        return len(
            self.scene_ids
        )

    def __getitem__(
        self,
        idx,
    ):
        sid = self.scene_ids[
            idx
        ]

        v = VISION_CACHE[
            sid
        ]

        t = scene_targets(
            sid,
            self.split,
        )

        return {
            "scene_id": sid,
            "pooled": v["pooled"].clone(),
            "patch9": v["patch9"].clone(),
            "regression": torch.tensor(
                t["regression"],
                dtype=torch.float32,
            ),
            "dominance": torch.tensor(
                t["dominance"],
                dtype=torch.long,
            ),
            "review": torch.tensor(
                t["review"],
                dtype=torch.float32,
            ),
        }

scene_train_loader = DataLoader(
    SceneDataset(
        train_scene_ids,
        "train",
    ),
    batch_size=64,
    shuffle=True,
    num_workers=0,
)

scene_val_loader = DataLoader(
    SceneDataset(
        val_scene_ids,
        "val",
    ),
    batch_size=34,
    shuffle=False,
    num_workers=0,
)

def pearson_corr_torch(
    x,
    y,
    eps=1e-8,
):
    x = x.float()
    y = y.float()

    x = x - x.mean()
    y = y - y.mean()

    num = (
        x * y
    ).sum()

    den = torch.sqrt(
        (x.square().sum() + eps)
        * (y.square().sum() + eps)
    )

    return (
        num
        / den
    ).clamp(
        -1.0,
        1.0,
    )

def pairwise_rank_loss(
    pred,
    target,
    minimum_target_gap=0.02,
    margin=0.01,
):
    pdiff = (
        pred[:, None]
        - pred[None, :]
    )

    tdiff = (
        target[:, None]
        - target[None, :]
    )

    valid = (
        tdiff.abs()
        >= minimum_target_gap
    )

    if not valid.any():
        return pred.new_tensor(0.0)

    sign = torch.sign(
        tdiff[valid]
    )

    return F.relu(
        margin
        - sign
        * pdiff[valid]
    ).mean()

def scene_losses(batch):
    pooled = batch[
        "pooled"
    ].to(
        DEVICE
    )

    patch9 = batch[
        "patch9"
    ].to(
        DEVICE
    )

    reg_target = batch[
        "regression"
    ].to(
        DEVICE
    )

    dom_target = batch[
        "dominance"
    ].to(
        DEVICE
    )

    review_target = batch[
        "review"
    ].to(
        DEVICE
    )

    out = gaussian_encoder(
        pooled,
        patch9,
        sample=True,
    )

    reg_loss = F.mse_loss(
        out["regression"],
        reg_target,
    )

    entropy_target = reg_target[
        :,
        1
    ]

    unc_mse = F.mse_loss(
        out["latent_uncertainty"],
        entropy_target,
    )

    corr = pearson_corr_torch(
        out["latent_uncertainty"],
        entropy_target,
    )

    corr_loss = (
        1.0
        - corr
    )

    rank_loss = pairwise_rank_loss(
        out["latent_uncertainty"],
        entropy_target,
    )

    dom_loss = F.cross_entropy(
        out["dominance_logits"],
        dom_target,
    )

    review_loss = F.binary_cross_entropy_with_logits(
        out["review_logit"],
        review_target,
    )

    mu = out[
        "mu"
    ]

    logvar = out[
        "logvar"
    ]

    kl = (
        -0.5
        * (
            1.0
            + logvar
            - mu.pow(2)
            - logvar.exp()
        )
    ).mean()

    total = (
        1.00 * reg_loss
        + 1.00 * unc_mse
        + 0.50 * corr_loss
        + 0.20 * rank_loss
        + 0.20 * dom_loss
        + 0.20 * review_loss
        + 1e-5 * kl
    )

    return {
        "loss": total,
        "reg_loss": reg_loss,
        "unc_mse": unc_mse,
        "corr": corr,
        "rank_loss": rank_loss,
        "dom_loss": dom_loss,
        "review_loss": review_loss,
        "kl": kl,
        "out": out,
        "reg_target": reg_target,
        "dom_target": dom_target,
        "review_target": review_target,
    }

print("P19-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C08 — FIXED SCENE-LEVEL GAUSSIAN PRETRAIN
# No hyperparameter sweep. Exactly 30 epochs.
# ============================================================
SCENE_EPOCHS = 30

scene_optimizer = torch.optim.AdamW(
    gaussian_encoder.parameters(),
    lr=5e-4,
    weight_decay=1e-4,
)

scene_history = []

@torch.inference_mode()
def evaluate_scene_encoder():
    gaussian_encoder.eval()

    pred_reg = []
    true_reg = []
    pred_unc = []
    pred_dom = []
    true_dom = []
    pred_review = []
    true_review = []

    for batch in scene_val_loader:
        pooled = batch["pooled"].to(DEVICE)
        patch9 = batch["patch9"].to(DEVICE)

        out = gaussian_encoder(
            pooled,
            patch9,
            sample=False,
        )

        pred_reg.append(
            out["regression"].cpu()
        )
        true_reg.append(
            batch["regression"]
        )

        pred_unc.append(
            out["latent_uncertainty"].cpu()
        )

        pred_dom.append(
            out["dominance_logits"].argmax(dim=1).cpu()
        )
        true_dom.append(
            batch["dominance"]
        )

        pred_review.append(
            (
                torch.sigmoid(
                    out["review_logit"]
                )
                >= 0.5
            ).long().cpu()
        )
        true_review.append(
            batch["review"].long()
        )

    pr = torch.cat(
        pred_reg
    ).numpy()

    tr = torch.cat(
        true_reg
    ).numpy()

    pu = torch.cat(
        pred_unc
    ).numpy()

    pdm = torch.cat(
        pred_dom
    ).numpy()

    tdm = torch.cat(
        true_dom
    ).numpy()

    prv = torch.cat(
        pred_review
    ).numpy()

    trv = torch.cat(
        true_review
    ).numpy()

    if (
        np.std(pu) > 1e-8
        and np.std(tr[:, 1]) > 1e-8
    ):
        entropy_r = float(
            np.corrcoef(
                pu,
                tr[:, 1],
            )[0, 1]
        )
    else:
        entropy_r = float("nan")

    return {
        "temp_fraction_MAE": float(
            np.mean(
                np.abs(
                    pr[:, 0]
                    - tr[:, 0]
                )
            )
        ),
        "entropy_aux_MAE": float(
            np.mean(
                np.abs(
                    pr[:, 1]
                    - tr[:, 1]
                )
            )
        ),
        "optical_share_MAE": float(
            np.mean(
                np.abs(
                    pr[:, 2]
                    - tr[:, 2]
                )
            )
        ),
        "latent_uncertainty_MAE": float(
            np.mean(
                np.abs(
                    pu
                    - tr[:, 1]
                )
            )
        ),
        "latent_entropy_correlation": entropy_r,
        "dominance_accuracy": float(
            np.mean(
                pdm == tdm
            )
        ),
        "review_accuracy": float(
            np.mean(
                prv == trv
            )
        ),
    }

for epoch in range(
    1,
    SCENE_EPOCHS + 1,
):
    gaussian_encoder.train()

    losses = []

    for batch in scene_train_loader:
        scene_optimizer.zero_grad(
            set_to_none=True
        )

        out = scene_losses(
            batch
        )

        assert torch.isfinite(
            out["loss"]
        ).all(), (
            f"Non-finite scene loss at epoch {epoch}"
        )

        out["loss"].backward()

        torch.nn.utils.clip_grad_norm_(
            gaussian_encoder.parameters(),
            1.0,
        )

        scene_optimizer.step()

        losses.append(
            float(
                out["loss"]
                .detach()
                .cpu()
            )
        )

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == SCENE_EPOCHS
    ):
        vm = evaluate_scene_encoder()

        scene_history.append({
            "epoch": epoch,
            "train_loss": float(
                np.mean(
                    losses
                )
            ),
            "validation": vm,
        })

        print(
            f"Epoch {epoch:02d}/{SCENE_EPOCHS} "
            f"train={np.mean(losses):.4f} "
            f"entropy_r={vm['latent_entropy_correlation']:.4f} "
            f"unc_MAE={vm['latent_uncertainty_MAE']:.4f} "
            f"dom_acc={vm['dominance_accuracy']:.3f} "
            f"review_acc={vm['review_accuracy']:.3f}"
        )

scene_val_metrics = evaluate_scene_encoder()

GAUSSIAN_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_09B_GaussianSceneEncoder.pt"
)

torch.save(
    {
        "state_dict": {
            k: v.detach().cpu()
            for k, v in gaussian_encoder.state_dict().items()
        },
        "scene_epochs": SCENE_EPOCHS,
        "validation": scene_val_metrics,
        "latent_dim": LATENT_DIM,
    },
    GAUSSIAN_PATH,
)

print("=" * 112)
print("P19-C08 — SCENE ENCODER COMPLETE")
print("=" * 112)

for k, v in scene_val_metrics.items():
    print(
        f"{k:<30}: {v:.6f}"
    )

print("Saved:", GAUSSIAN_PATH)
print("P19-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C09 — FREEZE GAUSSIAN SEMANTICS + BUILD VISUAL TASKS
# ============================================================
gaussian_encoder.eval()

for p in gaussian_encoder.parameters():
    p.requires_grad = False

def extent_bin(temp_fraction):
    x = float(temp_fraction)

    if x < 0.01:
        return "minimal"
    if x < 0.05:
        return "low"
    if x < 0.20:
        return "moderate"
    return "extensive"

def uncertainty_bin(entropy):
    x = float(entropy)

    if x < 0.05:
        return "low"
    if x < 0.15:
        return "moderate"
    return "high"

def visual_tasks_for_scene(
    scene_id,
    split,
):
    t = scene_targets(
        scene_id,
        split,
    )

    temp_fraction = float(
        t["regression"][0]
    )

    entropy = float(
        t["regression"][1]
    )

    dominance = ID_TO_DOM[
        int(
            t["dominance"]
        )
    ]

    review = (
        "yes"
        if float(
            t["review"]
        ) >= 0.5
        else "no"
    )

    return [
        {
            "id": f"{scene_id}__visual_extent",
            "scene_id": scene_id,
            "task": "visual_extent",
            "response_mode": "answer",
            "is_visual_only": True,
            "instruction": (
                "Using only the visual evidence, classify the "
                "temporary-flood extent. Return exactly "
                "EXTENT=minimal, EXTENT=low, EXTENT=moderate, "
                "or EXTENT=extensive."
            ),
            "response": (
                f"EXTENT={extent_bin(temp_fraction)}"
            ),
        },
        {
            "id": f"{scene_id}__visual_uncertainty",
            "scene_id": scene_id,
            "task": "visual_uncertainty",
            "response_mode": "answer",
            "is_visual_only": True,
            "instruction": (
                "Using only the visual evidence, classify the "
                "scene-level uncertainty descriptor. Return exactly "
                "UNCERTAINTY=low, UNCERTAINTY=moderate, "
                "or UNCERTAINTY=high."
            ),
            "response": (
                f"UNCERTAINTY={uncertainty_bin(entropy)}"
            ),
        },
        {
            "id": f"{scene_id}__visual_dominance",
            "scene_id": scene_id,
            "task": "visual_dominance",
            "response_mode": "answer",
            "is_visual_only": True,
            "instruction": (
                "Using only the visual evidence, classify the "
                "counterfactual sensor-dependence dominance. Return "
                "exactly DOMINANCE=optical, DOMINANCE=sar, "
                "or DOMINANCE=balanced."
            ),
            "response": (
                f"DOMINANCE={dominance}"
            ),
        },
        {
            "id": f"{scene_id}__visual_review",
            "scene_id": scene_id,
            "task": "visual_review",
            "response_mode": "answer",
            "is_visual_only": True,
            "instruction": (
                "Using only the visual evidence, classify whether "
                "the fixed FloodShift review policy flags this scene. "
                "Return exactly REVIEW=yes or REVIEW=no."
            ),
            "response": (
                f"REVIEW={review}"
            ),
        },
    ]

train_visual_examples = [
    ex
    for sid in train_scene_ids
    for ex in visual_tasks_for_scene(
        sid,
        "train",
    )
]

val_visual_examples = [
    ex
    for sid in val_scene_ids
    for ex in visual_tasks_for_scene(
        sid,
        "val",
    )
]

train_grounded_examples = [
    {
        **x,
        "is_visual_only": False,
    }
    for x in train_examples
]

val_grounded_examples = [
    {
        **x,
        "is_visual_only": False,
    }
    for x in val_examples
]

train_all_examples = (
    train_grounded_examples
    + train_visual_examples
)

val_all_examples = (
    val_grounded_examples
    + val_visual_examples
)

random.Random(
    SEED
).shuffle(
    train_all_examples
)

print("=" * 112)
print("P19-C09 — LANGUAGE CORPUS")
print("=" * 112)
print("Grounded train       :", len(train_grounded_examples))
print("Visual-only train    :", len(train_visual_examples))
print("Total train          :", len(train_all_examples))
print("Grounded validation  :", len(val_grounded_examples))
print("Visual-only validation:", len(val_visual_examples))
print("Total validation     :", len(val_all_examples))
print("Gaussian encoder     : FROZEN")
print("P19-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C10 — TOKENIZER + LANGUAGE DATASET
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)

MAX_SOURCE_LENGTH = 320
MAX_TARGET_LENGTH = 192

VISUAL_ONLY_PREFIX = (
    "You are FloodShift's visual-grounding classifier. "
    "For this task there is no structured scene-fact text. "
    "Use the supplied visual evidence tokens only. "
    "The requested category is descriptive and is not an emergency-risk threshold."
)

def source_text(example):
    if example["is_visual_only"]:
        return (
            VISUAL_ONLY_PREFIX
            + "\nQUESTION: "
            + example["instruction"]
        )

    return build_grounded_source(
        example
    )

class LanguageDataset(Dataset):
    def __init__(
        self,
        examples,
        split,
    ):
        self.examples = examples
        self.split = split

        self.shuffle_map = (
            TRAIN_SHUFFLE_MAP
            if split == "train"
            else VAL_SHUFFLE_MAP
        )

    def __len__(self):
        return len(
            self.examples
        )

    def __getitem__(
        self,
        idx,
    ):
        ex = self.examples[
            idx
        ]

        sid = ex[
            "scene_id"
        ]

        shuffled_sid = self.shuffle_map[
            sid
        ]

        v = VISION_CACHE[
            sid
        ]

        sv = VISION_CACHE[
            shuffled_sid
        ]

        return {
            "id": ex["id"],
            "scene_id": sid,
            "task": ex["task"],
            "response_mode": ex["response_mode"],
            "is_visual_only": bool(
                ex["is_visual_only"]
            ),
            "source_text": source_text(
                ex
            ),
            "target_text": ex[
                "response"
            ],
            "pooled": v[
                "pooled"
            ].clone(),
            "patch9": v[
                "patch9"
            ].clone(),
            "shuffled_pooled": sv[
                "pooled"
            ].clone(),
            "shuffled_patch9": sv[
                "patch9"
            ].clone(),
        }

class LanguageCollator:
    def __call__(
        self,
        rows,
    ):
        sources = [
            x["source_text"]
            for x in rows
        ]

        targets = [
            x["target_text"]
            for x in rows
        ]

        enc = tokenizer(
            sources,
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
            return_tensors="pt",
        )

        dec = tokenizer(
            text_target=targets,
            padding=True,
            truncation=True,
            max_length=MAX_TARGET_LENGTH,
            return_tensors="pt",
        )

        labels = dec[
            "input_ids"
        ]

        labels[
            labels
            == tokenizer.pad_token_id
        ] = -100

        return {
            "ids": [
                x["id"]
                for x in rows
            ],
            "scene_ids": [
                x["scene_id"]
                for x in rows
            ],
            "tasks": [
                x["task"]
                for x in rows
            ],
            "response_modes": [
                x["response_mode"]
                for x in rows
            ],
            "is_visual_only": torch.tensor(
                [
                    x["is_visual_only"]
                    for x in rows
                ],
                dtype=torch.bool,
            ),
            "source_texts": sources,
            "target_texts": targets,
            "input_ids": enc[
                "input_ids"
            ],
            "attention_mask": enc[
                "attention_mask"
            ],
            "labels": labels,
            "pooled": torch.stack(
                [
                    x["pooled"]
                    for x in rows
                ]
            ),
            "patch9": torch.stack(
                [
                    x["patch9"]
                    for x in rows
                ]
            ),
            "shuffled_pooled": torch.stack(
                [
                    x["shuffled_pooled"]
                    for x in rows
                ]
            ),
            "shuffled_patch9": torch.stack(
                [
                    x["shuffled_patch9"]
                    for x in rows
                ]
            ),
        }

collator = LanguageCollator()

train_language_loader = DataLoader(
    LanguageDataset(
        train_all_examples,
        "train",
    ),
    batch_size=2,
    shuffle=True,
    num_workers=0,
    collate_fn=collator,
    generator=torch.Generator().manual_seed(
        SEED
    ),
)

val_language_loader = DataLoader(
    LanguageDataset(
        val_all_examples,
        "val",
    ),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collator,
)

print("P19-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C11 — FLAN-T5 LoRA + SPATIAL VISUAL TOKEN ADAPTER
# ============================================================
text_model = AutoModelForSeq2SeqLM.from_pretrained(
    TEXT_MODEL_NAME
)

text_model.config.use_cache = False

if hasattr(
    text_model,
    "gradient_checkpointing_enable",
):
    text_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q",
        "v",
    ],
    bias="none",
)

text_model = get_peft_model(
    text_model,
    lora_config,
).to(DEVICE)

TEXT_DIM = int(
    text_model.config.d_model
)

class SpatialVisualTokenAdapter(nn.Module):
    def __init__(
        self,
        vision_dim,
        latent_dim,
        text_dim,
    ):
        super().__init__()

        self.patch_proj = nn.Sequential(
            nn.LayerNorm(
                vision_dim
            ),
            nn.Linear(
                vision_dim,
                text_dim,
            ),
        )

        self.z_proj = nn.Sequential(
            nn.LayerNorm(
                latent_dim
            ),
            nn.Linear(
                latent_dim,
                text_dim,
            ),
        )

        self.out_norm = nn.LayerNorm(
            text_dim
        )

        self.gate = nn.Sequential(
            nn.Linear(
                latent_dim,
                text_dim,
            ),
            nn.Sigmoid(),
        )

    def forward(
        self,
        patch9,
        z,
    ):
        p = self.patch_proj(
            patch9.float()
        )

        zt = self.z_proj(
            z.float()
        ).unsqueeze(1)

        gate = self.gate(
            z.float()
        ).unsqueeze(1)

        # Spatial patch content + Gaussian scene conditioning.
        tokens = (
            p * (
                0.5
                + gate
            )
            + zt
        )

        return self.out_norm(
            tokens
        )

visual_adapter = SpatialVisualTokenAdapter(
    VISION_DIM,
    LATENT_DIM,
    TEXT_DIM,
).to(DEVICE)

print("=" * 112)
print("P19-C11 — LANGUAGE MODEL")
print("=" * 112)
print("T5 d_model       :", TEXT_DIM)
print("Visual tokens    :", PATCH_COUNT)
print(
    "LoRA trainable   :",
    f"{sum(p.numel() for p in text_model.parameters() if p.requires_grad):,}"
)
print(
    "Adapter trainable:",
    f"{sum(p.numel() for p in visual_adapter.parameters() if p.requires_grad):,}"
)
print("Gaussian encoder : FROZEN")
print("P19-C11 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C12 — MULTIMODAL FORWARD
# ============================================================
@torch.no_grad()
def gaussian_forward(
    pooled,
    patch9,
    sample,
):
    gaussian_encoder.eval()

    return gaussian_encoder(
        pooled,
        patch9,
        sample=sample,
    )

def multimodal_inputs(
    batch,
    vision_mode,
    sample_latent,
):
    input_ids = batch[
        "input_ids"
    ].to(
        DEVICE
    )

    attention_mask = batch[
        "attention_mask"
    ].to(
        DEVICE
    )

    if vision_mode == "full":
        pooled = batch[
            "pooled"
        ].to(
            DEVICE
        )

        patch9 = batch[
            "patch9"
        ].to(
            DEVICE
        )

    elif vision_mode == "shuffled":
        pooled = batch[
            "shuffled_pooled"
        ].to(
            DEVICE
        )

        patch9 = batch[
            "shuffled_patch9"
        ].to(
            DEVICE
        )

    elif vision_mode == "zero":
        pooled = batch[
            "pooled"
        ].to(
            DEVICE
        )

        patch9 = batch[
            "patch9"
        ].to(
            DEVICE
        )

    else:
        raise ValueError(
            vision_mode
        )

    gout = gaussian_forward(
        pooled,
        patch9,
        sample=sample_latent,
    )

    visual_tokens = visual_adapter(
        patch9,
        gout["z"],
    )

    if vision_mode == "zero":
        visual_tokens = torch.zeros_like(
            visual_tokens
        )

    text_embeddings = (
        text_model
        .get_input_embeddings()(
            input_ids
        )
    )

    inputs_embeds = torch.cat(
        [
            visual_tokens,
            text_embeddings,
        ],
        dim=1,
    )

    visual_mask = torch.ones(
        (
            input_ids.shape[0],
            PATCH_COUNT,
        ),
        dtype=attention_mask.dtype,
        device=attention_mask.device,
    )

    multimodal_mask = torch.cat(
        [
            visual_mask,
            attention_mask,
        ],
        dim=1,
    )

    return (
        inputs_embeds,
        multimodal_mask,
        gout,
    )

def language_loss(
    batch,
    vision_mode="full",
    sample_latent=True,
):
    labels = batch[
        "labels"
    ].to(
        DEVICE
    )

    inputs_embeds, mask, gout = multimodal_inputs(
        batch,
        vision_mode=vision_mode,
        sample_latent=sample_latent,
    )

    out = text_model(
        inputs_embeds=inputs_embeds,
        attention_mask=mask,
        labels=labels,
        return_dict=True,
    )

    return out.loss

# Mandatory FP32 smoke test.
smoke = next(
    iter(
        train_language_loader
    )
)

text_model.eval()
visual_adapter.eval()

with torch.no_grad():
    smoke_loss = language_loss(
        smoke,
        vision_mode="full",
        sample_latent=True,
    )

assert torch.isfinite(
    smoke_loss
).all(), (
    f"Non-finite P19-C12 smoke loss: {smoke_loss}"
)

print("=" * 112)
print("P19-C12 — FP32 SMOKE TEST")
print("=" * 112)
print("Loss:", float(smoke_loss.cpu()))
print("Finite: PASS")
print("P19-C12 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C13 — FIXED 3-EPOCH LANGUAGE TRAINING
# ============================================================
LANGUAGE_EPOCHS = 3
GRAD_ACCUM = 8

optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p
                for p in text_model.parameters()
                if p.requires_grad
            ],
            "lr": 1e-4,
        },
        {
            "params": visual_adapter.parameters(),
            "lr": 3e-4,
        },
    ],
    weight_decay=1e-4,
)

steps_per_epoch = math.ceil(
    len(train_language_loader)
    / GRAD_ACCUM
)

total_steps = (
    steps_per_epoch
    * LANGUAGE_EPOCHS
)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(
        1,
        int(
            0.10
            * total_steps
        ),
    ),
    num_training_steps=total_steps,
)

BEST_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_09B_best_language_adapter.pt"
)

@torch.inference_mode()
def validation_text_loss(
    vision_mode="full",
):
    text_model.eval()
    visual_adapter.eval()

    total = 0.0
    count = 0

    grounded_total = 0.0
    grounded_count = 0

    visual_total = 0.0
    visual_count = 0

    for batch in val_language_loader:
        loss = language_loss(
            batch,
            vision_mode=vision_mode,
            sample_latent=False,
        )

        b = len(
            batch["ids"]
        )

        total += float(
            loss.cpu()
        ) * b
        count += b

        # Separate loaders would be more exact for token weighting,
        # but batch-level subdivision is sufficient for diagnostic loss.
        flags = batch[
            "is_visual_only"
        ]

        n_visual = int(
            flags.sum()
        )

        n_grounded = (
            b
            - n_visual
        )

        if n_grounded > 0:
            grounded_total += float(
                loss.cpu()
            ) * n_grounded
            grounded_count += n_grounded

        if n_visual > 0:
            visual_total += float(
                loss.cpu()
            ) * n_visual
            visual_count += n_visual

    return {
        "all": total / max(count, 1),
        "grounded": grounded_total / max(grounded_count, 1),
        "visual_only": visual_total / max(visual_count, 1),
    }

history = []
best_val = float("inf")
best_epoch = None

for epoch in range(
    1,
    LANGUAGE_EPOCHS + 1,
):
    t0 = time.time()

    text_model.train()
    visual_adapter.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    running = []

    for bi, batch in enumerate(
        train_language_loader,
        start=1,
    ):
        loss = language_loss(
            batch,
            vision_mode="full",
            sample_latent=True,
        )

        assert torch.isfinite(
            loss
        ).all(), (
            f"Non-finite language loss at epoch={epoch}, batch={bi}"
        )

        (
            loss
            / GRAD_ACCUM
        ).backward()

        running.append(
            float(
                loss.detach().cpu()
            )
        )

        should_step = (
            bi % GRAD_ACCUM == 0
            or bi == len(train_language_loader)
        )

        if should_step:
            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p in list(
                        text_model.parameters()
                    )
                    + list(
                        visual_adapter.parameters()
                    )
                    if p.requires_grad
                ],
                1.0,
            )

            optimizer.step()
            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        if (
            bi == 1
            or bi % 250 == 0
            or bi == len(train_language_loader)
        ):
            print(
                f"Epoch {epoch}/{LANGUAGE_EPOCHS} "
                f"[{bi:04d}/{len(train_language_loader)}] "
                f"loss={running[-1]:.4f}"
            )

    vm = validation_text_loss(
        "full"
    )

    row = {
        "epoch": epoch,
        "train_loss": float(
            np.mean(
                running
            )
        ),
        "validation": vm,
        "minutes": (
            time.time()
            - t0
        ) / 60.0,
    }

    history.append(
        row
    )

    print("=" * 112)
    print(
        f"EPOCH {epoch}/{LANGUAGE_EPOCHS} COMPLETE"
    )
    print(
        "Train loss:",
        row["train_loss"]
    )
    print(
        "Val loss  :",
        vm
    )
    print(
        "Minutes   :",
        round(
            row["minutes"],
            2,
        )
    )

    if vm[
        "all"
    ] < best_val:
        best_val = float(
            vm["all"]
        )

        best_epoch = int(
            epoch
        )

        torch.save(
            {
                "epoch": best_epoch,
                "val_loss": best_val,
                "peft_state_dict": {
                    k: v.detach().cpu()
                    for k, v in get_peft_model_state_dict(
                        text_model
                    ).items()
                },
                "visual_adapter_state_dict": {
                    k: v.detach().cpu()
                    for k, v in visual_adapter.state_dict().items()
                },
            },
            BEST_PATH,
        )

        print(
            "✅ New best:",
            BEST_PATH
        )

    print("=" * 112)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Best epoch:", best_epoch)
print("Best validation loss:", best_val)
print("P19-C13 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C14 — RELOAD BEST + TRUE VISUAL ABLATION
# ============================================================
best = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

set_peft_model_state_dict(
    text_model,
    best["peft_state_dict"],
)

visual_adapter.load_state_dict(
    best["visual_adapter_state_dict"],
    strict=True,
)

text_model = text_model.to(
    DEVICE
).eval()

visual_adapter = visual_adapter.to(
    DEVICE
).eval()

full_loss = validation_text_loss(
    "full"
)

zero_loss = validation_text_loss(
    "zero"
)

shuffle_loss = validation_text_loss(
    "shuffled"
)

ablation_df = pd.DataFrame([
    {
        "mode": "full",
        **full_loss,
    },
    {
        "mode": "zero",
        **zero_loss,
    },
    {
        "mode": "true_scene_shuffled",
        **shuffle_loss,
    },
])

display(
    ablation_df
)

print(
    "All-task full→zero loss increase:",
    zero_loss["all"] - full_loss["all"]
)

print(
    "All-task full→shuffle loss increase:",
    shuffle_loss["all"] - full_loss["all"]
)

print(
    "Visual-only full→zero loss increase:",
    zero_loss["visual_only"] - full_loss["visual_only"]
)

print(
    "Visual-only full→shuffle loss increase:",
    shuffle_loss["visual_only"] - full_loss["visual_only"]
)

print("P19-C14 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C15 — DETERMINISTIC GENERATION
# ============================================================
@torch.inference_mode()
def generate_batch(
    batch,
    vision_mode="full",
    max_new_tokens=192,
):
    text_model.eval()
    visual_adapter.eval()

    inputs_embeds, mask, _ = multimodal_inputs(
        batch,
        vision_mode=vision_mode,
        sample_latent=False,
    )

    generated = text_model.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
    )

    return tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
    )

def generate_validation(
    vision_mode="full",
):
    rows = []

    for bi, batch in enumerate(
        val_language_loader,
        start=1,
    ):
        preds = generate_batch(
            batch,
            vision_mode=vision_mode,
        )

        for i, pred in enumerate(
            preds
        ):
            rows.append({
                "id": batch["ids"][i],
                "scene_id": batch["scene_ids"][i],
                "task": batch["tasks"][i],
                "response_mode": batch["response_modes"][i],
                "is_visual_only": bool(
                    batch["is_visual_only"][i]
                ),
                "source_text": batch["source_texts"][i],
                "reference": batch["target_texts"][i],
                "prediction": pred,
                "vision_mode": vision_mode,
            })

        if (
            bi == 1
            or bi % 30 == 0
            or bi == len(val_language_loader)
        ):
            print(
                f"{vision_mode}: [{bi:03d}/{len(val_language_loader)}]"
            )

    return pd.DataFrame(
        rows
    )

full_generation = generate_validation(
    "full"
)

zero_generation = generate_validation(
    "zero"
)

shuffle_generation = generate_validation(
    "shuffled"
)

assert len(
    full_generation
) == 476

print("P19-C15 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C16 — VISUAL-ONLY GROUNDING ACCURACY
# ============================================================
def normalize_text(text):
    return re.sub(
        r"\s+",
        " ",
        str(text).strip().lower(),
    )

def extract_visual_label(
    task,
    text,
):
    t = normalize_text(
        text
    )

    patterns = {
        "visual_extent": (
            r"extent\s*=\s*(minimal|low|moderate|extensive)"
        ),
        "visual_uncertainty": (
            r"uncertainty\s*=\s*(low|moderate|high)"
        ),
        "visual_dominance": (
            r"dominance\s*=\s*(optical|sar|balanced)"
        ),
        "visual_review": (
            r"review\s*=\s*(yes|no)"
        ),
    }

    m = re.search(
        patterns[task],
        t,
    )

    return (
        m.group(1)
        if m
        else None
    )

def visual_accuracy(df):
    rows = df[
        df["is_visual_only"]
    ].copy()

    correct = []

    for _, row in rows.iterrows():
        task = row[
            "task"
        ]

        ref = extract_visual_label(
            task,
            row["reference"],
        )

        pred = extract_visual_label(
            task,
            row["prediction"],
        )

        correct.append(
            ref is not None
            and pred == ref
        )

    rows[
        "correct"
    ] = correct

    overall = float(
        rows["correct"].mean()
    )

    per_task = (
        rows
        .groupby(
            "task",
            as_index=False,
        )
        .agg(
            examples=("id", "count"),
            accuracy=("correct", "mean"),
        )
    )

    return (
        overall,
        per_task,
    )

full_visual_acc, full_visual_task = visual_accuracy(
    full_generation
)

zero_visual_acc, zero_visual_task = visual_accuracy(
    zero_generation
)

shuffle_visual_acc, shuffle_visual_task = visual_accuracy(
    shuffle_generation
)

print("=" * 112)
print("P19-C16 — VISUAL-ONLY ACCURACY")
print("=" * 112)
print("Full     :", full_visual_acc)
print("Zero     :", zero_visual_acc)
print("Shuffled :", shuffle_visual_acc)
print()
print("Full task-level:")
display(
    full_visual_task
)

print(
    "Full-zero gap    :",
    full_visual_acc
    - zero_visual_acc
)

print(
    "Full-shuffle gap :",
    full_visual_acc
    - shuffle_visual_acc
)

print("P19-C16 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C17 — GROUNDED GENERATION METRICS
# Original 340 validation instructions only.
# ============================================================
NUMBER_RE = re.compile(
    r"[-+]?\d+(?:\.\d+)?"
)

def token_f1(
    reference,
    prediction,
):
    ref = normalize_text(
        reference
    ).split()

    pred = normalize_text(
        prediction
    ).split()

    if not ref and not pred:
        return 1.0
    if not ref or not pred:
        return 0.0

    rc = Counter(
        ref
    )
    pc = Counter(
        pred
    )

    overlap = sum(
        (
            rc
            & pc
        ).values()
    )

    p = overlap / max(
        len(pred),
        1,
    )

    r = overlap / max(
        len(ref),
        1,
    )

    if p + r == 0:
        return 0.0

    return (
        2
        * p
        * r
        / (
            p
            + r
        )
    )

def extract_numbers(text):
    return [
        float(x)
        for x in NUMBER_RE.findall(
            str(text)
        )
    ]

def number_match(
    x,
    candidates,
):
    return any(
        math.isclose(
            x,
            y,
            rel_tol=0.0,
            abs_tol=0.011,
        )
        for y in candidates
    )

def numeric_recall(
    reference,
    prediction,
):
    ref = extract_numbers(
        reference
    )

    pred = extract_numbers(
        prediction
    )

    if not ref:
        return 1.0

    return (
        sum(
            number_match(
                x,
                pred,
            )
            for x in ref
        )
        / len(ref)
    )

def unsupported_numeric_rate(
    source,
    prediction,
):
    allowed = extract_numbers(
        source
    )

    pred = extract_numbers(
        prediction
    )

    if not pred:
        return 0.0

    return (
        sum(
            not number_match(
                x,
                allowed,
            )
            for x in pred
        )
        / len(pred)
    )

REFUSAL_MARKERS = [
    "does not",
    "cannot",
    "not contain",
    "not establish",
    "not provide",
    "cannot be determined",
    "would require",
]

def looks_like_refusal(text):
    t = normalize_text(
        text
    )

    return any(
        marker in t
        for marker in REFUSAL_MARKERS
    )

grounded = full_generation[
    ~full_generation[
        "is_visual_only"
    ]
].copy()

metric_rows = []

for _, row in grounded.iterrows():
    metric_rows.append({
        "id": row["id"],
        "scene_id": row["scene_id"],
        "task": row["task"],
        "response_mode": row["response_mode"],
        "token_f1": token_f1(
            row["reference"],
            row["prediction"],
        ),
        "numeric_recall": numeric_recall(
            row["reference"],
            row["prediction"],
        ),
        "unsupported_numeric_rate": unsupported_numeric_rate(
            row["source_text"],
            row["prediction"],
        ),
        "looks_like_refusal": looks_like_refusal(
            row["prediction"]
        ),
    })

grounded_metrics = pd.DataFrame(
    metric_rows
)

refuse_rows = grounded_metrics[
    grounded_metrics[
        "response_mode"
    ]
    == "refuse"
]

answer_rows = grounded_metrics[
    grounded_metrics[
        "response_mode"
    ]
    == "answer"
]

grounded_summary = {
    "token_f1": float(
        grounded_metrics[
            "token_f1"
        ].mean()
    ),
    "numeric_recall": float(
        grounded_metrics[
            "numeric_recall"
        ].mean()
    ),
    "unsupported_numeric_rate": float(
        grounded_metrics[
            "unsupported_numeric_rate"
        ].mean()
    ),
    "refusal_recall": float(
        refuse_rows[
            "looks_like_refusal"
        ].mean()
    ),
    "false_refusal_rate_on_answers": float(
        answer_rows[
            "looks_like_refusal"
        ].mean()
    ),
}

task_metrics = (
    grounded_metrics
    .groupby(
        "task",
        as_index=False,
    )
    .agg(
        examples=("id", "count"),
        token_f1=("token_f1", "mean"),
        numeric_recall=("numeric_recall", "mean"),
        unsupported_numeric_rate=("unsupported_numeric_rate", "mean"),
    )
)

print("=" * 112)
print("P19-C17 — GROUNDED METRICS")
print("=" * 112)

for k, v in grounded_summary.items():
    print(
        f"{k:<32}: {v:.6f}"
    )

print()
display(
    task_metrics
)

print("P19-C17 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C18 — QUALITATIVE CHECKS
# ============================================================
sample_tasks = [
    "extent",
    "uncertainty",
    "sensor_attribution",
    "temporary_sensor_attribution",
    "spatial_geometry",
    "integrated_summary",
    "review_policy",
    "cause_refusal",
    "impact_refusal",
    "forecast_refusal",
    "visual_extent",
    "visual_uncertainty",
    "visual_dominance",
    "visual_review",
]

for task in sample_tasks:
    row = full_generation[
        full_generation[
            "task"
        ]
        == task
    ].iloc[
        0
    ]

    print("=" * 112)
    print("TASK :", task)
    print("SCENE:", row["scene_id"])
    print("REFERENCE:")
    print(row["reference"])
    print()
    print("PREDICTION:")
    print(row["prediction"])
    print()

print("P19-C18 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C19 — DECLARED ACCEPTANCE GATES
# ============================================================
gates = {
    "latent_entropy_correlation_ge_0.30": (
        scene_val_metrics[
            "latent_entropy_correlation"
        ]
        >= 0.30
    ),
    "visual_full_accuracy_ge_0.70": (
        full_visual_acc
        >= 0.70
    ),
    "visual_full_minus_zero_ge_0.15": (
        (
            full_visual_acc
            - zero_visual_acc
        )
        >= 0.15
    ),
    "visual_full_minus_shuffle_ge_0.15": (
        (
            full_visual_acc
            - shuffle_visual_acc
        )
        >= 0.15
    ),
    "numeric_recall_ge_0.85": (
        grounded_summary[
            "numeric_recall"
        ]
        >= 0.85
    ),
    "unsupported_numbers_le_0.05": (
        grounded_summary[
            "unsupported_numeric_rate"
        ]
        <= 0.05
    ),
    "refusal_recall_ge_0.95": (
        grounded_summary[
            "refusal_recall"
        ]
        >= 0.95
    ),
    "false_refusal_le_0.05": (
        grounded_summary[
            "false_refusal_rate_on_answers"
        ]
        <= 0.05
    ),
}

gate_df = pd.DataFrame(
    [
        {
            "gate": k,
            "pass": bool(v),
        }
        for k, v in gates.items()
    ]
)

display(
    gate_df
)

all_pass = bool(
    all(
        gates.values()
    )
)

print()
print(
    "09B STATUS:",
    (
        "ACCEPTED"
        if all_pass
        else "NOT_ACCEPTED"
    ),
)

print(
    "Frozen geographic test:",
    "LOCKED_NOT_EVALUATED",
)

print("P19-C19 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C20 — EXPORT
# ============================================================
# Save compact model artifacts.
LORA_DIR = (
    CHECKPOINT_ROOT
    / "flan_t5_lora"
)

text_model.save_pretrained(
    LORA_DIR
)

TOKENIZER_DIR = (
    CHECKPOINT_ROOT
    / "tokenizer"
)

tokenizer.save_pretrained(
    TOKENIZER_DIR
)

VISUAL_ADAPTER_PATH = (
    CHECKPOINT_ROOT
    / "FloodShift_09B_SpatialVisualTokenAdapter.pt"
)

torch.save(
    {
        "state_dict": {
            k: v.detach().cpu()
            for k, v in visual_adapter.state_dict().items()
        },
        "vision_model": VISION_MODEL_NAME,
        "text_model": TEXT_MODEL_NAME,
        "vision_dim": VISION_DIM,
        "latent_dim": LATENT_DIM,
        "patch_tokens": PATCH_COUNT,
        "best_epoch": best_epoch,
    },
    VISUAL_ADAPTER_PATH,
)

FULL_GEN_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_full_generation.csv"
)
ZERO_GEN_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_zero_generation.csv"
)
SHUFFLE_GEN_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_shuffle_generation.csv"
)
GROUNDED_METRIC_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_grounded_metrics.csv"
)
TASK_METRIC_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_task_metrics.csv"
)
ABLATION_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_visual_ablation.csv"
)
GATES_CSV = (
    METRIC_ROOT
    / "FloodShift_09B_acceptance_gates.csv"
)
HISTORY_JSON = (
    METRIC_ROOT
    / "FloodShift_09B_history.json"
)
SCENE_HISTORY_JSON = (
    METRIC_ROOT
    / "FloodShift_09B_scene_history.json"
)

full_generation.to_csv(
    FULL_GEN_CSV,
    index=False,
)
zero_generation.to_csv(
    ZERO_GEN_CSV,
    index=False,
)
shuffle_generation.to_csv(
    SHUFFLE_GEN_CSV,
    index=False,
)
grounded_metrics.to_csv(
    GROUNDED_METRIC_CSV,
    index=False,
)
task_metrics.to_csv(
    TASK_METRIC_CSV,
    index=False,
)
ablation_df.to_csv(
    ABLATION_CSV,
    index=False,
)
gate_df.to_csv(
    GATES_CSV,
    index=False,
)

with open(
    HISTORY_JSON,
    "w",
) as f:
    json.dump(
        history,
        f,
        indent=2,
    )

with open(
    SCENE_HISTORY_JSON,
    "w",
) as f:
    json.dump(
        scene_history,
        f,
        indent=2,
    )

PROTOCOL_JSON = (
    OUTPUT_ROOT
    / "FloodShift_09B_protocol_record.json"
)

protocol09b = {
    "experiment": "FloodShift-09B-Controlled-Variational-VLM-Refinement",
    "source_08b_sha256": EXPECTED_08B_SHA,
    "baseline_09a_sha256": BASELINE_09A_SHA,
    "source_segmenter": "FloodShift-04H-CAL",
    "changes_from_09a": [
        "CLIP spatial patch tokens instead of pooled-only conditioning",
        "scene-level Gaussian pretraining with correlation and ranking supervision",
        "Gaussian scene encoder frozen before language training",
        "task-specific deterministic evidence",
        "four visual-only grounding tasks",
        "true scene-level shuffled-vision derangement",
    ],
    "scene_encoder_validation": scene_val_metrics,
    "language_best_epoch": best_epoch,
    "language_best_validation_loss": best_val,
    "visual_grounding": {
        "full_accuracy": full_visual_acc,
        "zero_accuracy": zero_visual_acc,
        "shuffled_accuracy": shuffle_visual_acc,
        "full_minus_zero": (
            full_visual_acc
            - zero_visual_acc
        ),
        "full_minus_shuffled": (
            full_visual_acc
            - shuffle_visual_acc
        ),
    },
    "grounded_generation": grounded_summary,
    "acceptance_gates": {
        k: bool(v)
        for k, v in gates.items()
    },
    "status": (
        "ACCEPTED"
        if all_pass
        else "NOT_ACCEPTED"
    ),
    "frozen_geographic_test": "LOCKED_NOT_EVALUATED",
}

with open(
    PROTOCOL_JSON,
    "w",
) as f:
    json.dump(
        protocol09b,
        f,
        indent=2,
    )

print("=" * 112)
print("P19-C20 — EXPORT")
print("=" * 112)
print("Status      :", protocol09b["status"])
print("Gaussian    :", GAUSSIAN_PATH)
print("Visual adap.:", VISUAL_ADAPTER_PATH)
print("LoRA        :", LORA_DIR)
print("Protocol    :", PROTOCOL_JSON)
print("Geo test    : LOCKED_NOT_EVALUATED")
print("P19-C20 COMPLETE ✅")


In [ ]:
# ============================================================
# P19-C21 — ARCHIVE + SHA256
# ============================================================
def file_sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(
                chunk
            )

    return h.hexdigest()

MANIFEST_PATH = (
    OUTPUT_ROOT
    / "FloodShift_09B_file_manifest.csv"
)

manifest_rows = []

for root in [
    CHECKPOINT_ROOT,
    METRIC_ROOT,
]:
    for p in sorted(
        root.rglob("*")
    ):
        if p.is_file():
            manifest_rows.append({
                "relative_path": str(
                    p.relative_to(
                        OUTPUT_ROOT
                    )
                ),
                "bytes": int(
                    p.stat().st_size
                ),
                "sha256": file_sha256(
                    p
                ),
            })

manifest_rows.append({
    "relative_path": str(
        PROTOCOL_JSON.relative_to(
            OUTPUT_ROOT
        )
    ),
    "bytes": int(
        PROTOCOL_JSON.stat().st_size
    ),
    "sha256": file_sha256(
        PROTOCOL_JSON
    ),
})

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
)

ARCHIVE_PATH = (
    OUTPUT_ROOT
    / "FloodShift_09B_ARCHIVE.zip"
)

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for root in [
        CHECKPOINT_ROOT,
        METRIC_ROOT,
    ]:
        for p in sorted(
            root.rglob("*")
        ):
            if p.is_file():
                zf.write(
                    p,
                    arcname=p.relative_to(
                        OUTPUT_ROOT
                    ),
                )

    zf.write(
        PROTOCOL_JSON,
        arcname=PROTOCOL_JSON.relative_to(
            OUTPUT_ROOT
        ),
    )

    zf.write(
        MANIFEST_PATH,
        arcname=MANIFEST_PATH.relative_to(
            OUTPUT_ROOT
        ),
    )

ARCHIVE_SHA = file_sha256(
    ARCHIVE_PATH
)

print("=" * 112)
print("P19-C21 — FINAL")
print("=" * 112)
print("09B status       :", "ACCEPTED" if all_pass else "NOT_ACCEPTED")
print("Entropy corr.    :", scene_val_metrics["latent_entropy_correlation"])
print("Visual full acc. :", full_visual_acc)
print("Visual zero acc. :", zero_visual_acc)
print("Visual shuffle   :", shuffle_visual_acc)
print("Numeric recall   :", grounded_summary["numeric_recall"])
print("Unsupported nums :", grounded_summary["unsupported_numeric_rate"])
print("Refusal recall   :", grounded_summary["refusal_recall"])
print("False refusal    :", grounded_summary["false_refusal_rate_on_answers"])
print("Geo test         : LOCKED_NOT_EVALUATED")
print("Archive          :", ARCHIVE_PATH)
print("Archive SHA256   :", ARCHIVE_SHA)
print()
print("Upload the EXECUTED notebook for acceptance analysis.")
print("P19-C21 COMPLETE ✅")


## What to send back

After the run, upload the **executed notebook**.

The most important cells are:

- **P19-C08** — Gaussian scene-encoder validation
- **P19-C13** — 3 language-training epochs
- **P19-C14** — correct full / zero / true-shuffled loss ablation
- **P19-C16** — visual-only grounding accuracy
- **P19-C17** — grounded-generation metrics
- **P19-C18** — qualitative outputs
- **P19-C19** — declared acceptance gates
- **P19-C21** — final archive SHA

Do **not** open the geographic test regardless of whether 09B passes. The verifier/abstention stage comes before final test evaluation.
